# OYKHCHAR LoRA Training on FLUX.1-dev

Training a custom character LoRA for consistent video generation.

**Character:** OYKHCHAR (minimalist white stick figure)
**Base Model:** FLUX.1-dev
**Training Images:** 26 high-quality examples
**Expected Time:** 30-45 minutes on T4 GPU

In [ ]:
# Install dependencies
!pip install -q diffusers[torch]==0.30.3 transformers accelerate peft safetensors sentencepiece protobuf

In [ ]:
# Import libraries
import torch
from diffusers import FluxPipeline
from peft import LoraConfig, get_peft_model
import os
from pathlib import Path
import zipfile

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Extract training images
dataset_path = '/kaggle/input/oykhchar-training/images.zip'
extract_path = '/kaggle/working/training_images'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(dataset_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# Count images
image_files = list(Path(extract_path).glob('**/*.jpg'))
print(f"✅ Extracted {len(image_files)} training images")
print(f"📁 Location: {extract_path}")

In [ ]:
# Load FLUX.1-dev base model
print("Loading FLUX.1-dev model...")
pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    torch_dtype=torch.bfloat16
)
pipe = pipe.to("cuda")
print("✅ Model loaded")

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
    lora_dropout=0.1,
)

# Apply LoRA to transformer
pipe.transformer = get_peft_model(pipe.transformer, lora_config)
print("✅ LoRA configured")
pipe.transformer.print_trainable_parameters()

In [ ]:
# Prepare training data
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

class LoRADataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_paths = list(Path(image_dir).glob('**/*.jpg'))
        self.transform = transform or transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
        
        # Load captions
        self.captions = []
        for img_path in self.image_paths:
            caption_path = img_path.with_suffix('.txt')
            if caption_path.exists():
                with open(caption_path, 'r') as f:
                    self.captions.append(f.read().strip())
            else:
                self.captions.append("OYKHCHAR character")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        image = self.transform(image)
        caption = self.captions[idx]
        return {'image': image, 'caption': caption}

dataset = LoRADataset(extract_path)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
print(f"✅ Dataset prepared: {len(dataset)} images")

In [ ]:
# Training loop
from torch.optim import AdamW
from tqdm import tqdm

optimizer = AdamW(pipe.transformer.parameters(), lr=1e-4)
num_epochs = 50
steps_per_epoch = len(dataloader)

print(f"🎓 Starting training: {num_epochs} epochs, {steps_per_epoch} steps/epoch")
print("="*50)

pipe.transformer.train()
global_step = 0

for epoch in range(num_epochs):
    epoch_loss = 0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for batch in progress_bar:
        images = batch['image'].to("cuda")
        captions = batch['caption']
        
        # Forward pass
        with torch.cuda.amp.autocast():
            # Encode text
            text_embeddings = pipe.encode_prompt(
                captions,
                device="cuda",
                num_images_per_prompt=1,
                do_classifier_free_guidance=False
            )
            
            # Simple reconstruction loss
            latents = pipe.vae.encode(images).latent_dist.sample()
            noise = torch.randn_like(latents)
            timesteps = torch.randint(0, 1000, (latents.shape[0],), device="cuda")
            
            noisy_latents = pipe.scheduler.add_noise(latents, noise, timesteps)
            model_pred = pipe.transformer(
                noisy_latents,
                timesteps,
                encoder_hidden_states=text_embeddings[0]
            ).sample
            
            loss = torch.nn.functional.mse_loss(model_pred, noise)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        global_step += 1
        
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
    
    avg_loss = epoch_loss / steps_per_epoch
    print(f"Epoch {epoch+1} completed - Avg Loss: {avg_loss:.4f}")
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        checkpoint_path = f"/kaggle/working/checkpoint_epoch_{epoch+1}.pt"
        pipe.transformer.save_pretrained(checkpoint_path)
        print(f"💾 Checkpoint saved: {checkpoint_path}")

print("\n✅ Training complete!")

In [ ]:
# Save final LoRA weights
output_dir = "/kaggle/working/oykhchar-lora-final"
pipe.transformer.save_pretrained(output_dir)
print(f"✅ LoRA weights saved: {output_dir}")

# List output files
!ls -lh /kaggle/working/oykhchar-lora-final/

In [ ]:
# Test the trained LoRA
print("🧪 Testing trained LoRA...")
pipe.transformer.eval()

test_prompts = [
    "OYKHCHAR character standing with arms raised in celebration",
    "OYKHCHAR character sitting and thinking",
    "OYKHCHAR character running forward energetically"
]

for i, prompt in enumerate(test_prompts):
    print(f"Generating test image {i+1}: {prompt}")
    image = pipe(
        prompt,
        num_inference_steps=28,
        guidance_scale=3.5,
        height=1024,
        width=1024
    ).images[0]
    
    image.save(f"/kaggle/working/test_{i+1}.png")
    print(f"✅ Saved: test_{i+1}.png")

print("\n🎉 All done! Download the LoRA weights from /kaggle/working/oykhchar-lora-final/")